# 유전 알고리즘 실습

**Genetic Algorithm · GA · 진화 알고리즘**

선택·교차·변이를 반복해 좋은 해를 찾아가는 최적화 방법.

소재 분야에서 이해하기: 층 구성과 두께 조합을 진화시켜 목표 반사율을 찾는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [pymoo 다목적 최적화 문서](https://pymoo.org/)

## 1. 다층 박막 두께 최적화

목표 반사율에 가까워지도록 5개 층의 두께를 진화시킵니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

target_profile = np.array([0.1, 0.4, 0.8, 0.4, 0.1])

def performance(thickness):
    """개념 설명용 가상 목적함수. 실제 광학 계산이 아닙니다."""
    response = np.sin(thickness / 40.0) ** 2
    return -np.mean((response - target_profile) ** 2)

print('무작위 해 성능 %.4f' % performance(rng.uniform(10, 200, 5)))

In [ ]:
def evolve(generations=60, population=60, seed=0):
    local = np.random.default_rng(seed)
    genes = local.uniform(10, 200, (population, 5))
    history = []
    for generation in range(generations):
        fitness = np.array([performance(gene) for gene in genes])
        order = np.argsort(-fitness)
        genes, fitness = genes[order], fitness[order]
        history.append(fitness[0])
        parents = genes[:population // 3]
        children = []
        while len(children) < population - len(parents):
            a, b = parents[local.integers(0, len(parents), 2)]
            mask = local.random(5) < 0.5
            child = np.where(mask, a, b)                      # 교차
            child = child + local.normal(0, 6, 5) * (local.random(5) < 0.3)   # 변이
            children.append(np.clip(child, 10, 200))
        genes = np.vstack([parents, np.array(children)])
    return genes[0], history

best, history = evolve()
plt.plot(history); plt.xlabel('generation'); plt.ylabel('best fitness'); plt.show()
print('최적 두께 %s' % np.round(best, 1))
print('최종 성능 %.5f' % performance(best))

## 2. 무작위 탐색과 비교

In [ ]:
random_best = max(performance(rng.uniform(10, 200, 5)) for _ in range(60 * 60))
print('같은 평가 횟수의 무작위 탐색 %.5f' % random_best)
print('유전 알고리즘             %.5f' % performance(best))
print('\n변이 폭이 너무 작으면 지역해에 갇히고, 너무 크면 수렴하지 않습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#genetic-algorithm)을 여세요.